# RL Token Autoencoder — Embedding Analysis

Analyze what the RLT autoencoder reconstructs well vs poorly, and what the compressed `z_rl` token actually captures.

- **Inputs:** dumped Pi0.5-CoT prefix embeddings `z_{1:M}` (+ optional per-row text/image metadata) under `rl_token_embeddings/<dataset>/{train,val}`.
- **Model:** a trained AE checkpoint under `rl_token_ae/<run>/ae_ckpt.pt`.
- **Flow:** load → per-sample metrics → *where*/​*what* drives loss → `z_rl` geometry & information content → raw-input inspection.

Run from the repo root on the machine that has the data + checkpoints.

## 1. Setup, paths, and load the model

In [ ]:
import glob, pathlib, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# --- Paths (edit RUN to the checkpoint you want to analyze) ---
REPO = pathlib.Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

EMB_DIR   = REPO / "rl_token_embeddings" / "traffic_light_v0"
VAL_DIR   = EMB_DIR / "val"
TRAIN_DIR = EMB_DIR / "train"

RUN  = "traffic_light_v0"                       # <-- subdir under rl_token_ae/
CKPT = REPO / "rl_token_ae" / RUN / "ae_ckpt.pt"

MAX_SAMPLES = None                              # cap loaded samples for a quick pass, or None for all
device = "cuda" if torch.cuda.is_available() else "cpu"

print("repo :", REPO)
print("ckpt :", CKPT, "| exists:", CKPT.exists())
print("val  :", VAL_DIR, "| exists:", VAL_DIR.exists())
print("device:", device)

In [ ]:
from openpi.models.rl_token import RLTokenAEConfig, RLTokenAutoencoder

ckpt = torch.load(CKPT, map_location=device)
cfg = RLTokenAEConfig(**ckpt["cfg"])
model = RLTokenAutoencoder(cfg).to(device).eval()
model.load_state_dict(ckpt["model"])
print("cfg:", cfg)
print("params: %.2fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

## 2. Load the dumped embeddings

Each `.npz` shard holds `prefix_out` (the per-token embeddings `z_{1:M}`), `prefix_mask`, `actions`, and the segment sizes. The compacted layout is `[vision | prompt | reasoning | subtask | fast]`.

In [ ]:
# Optional per-row raw inputs (present only if the shards were dumped with --save-metadata).
META_KEYS = ["tokenized_prompt", "tokenized_prompt_mask", "tokenized_reasoning", "tokenized_reasoning_mask",
             "tokenized_subtask", "tokenized_subtask_mask", "tokenized_fast", "tokenized_fast_mask",
             "base_image", "base_image_jpeg", "base_image_jpeg_lengths"]

def load_shards(shard_dir, max_samples=None):
    paths = sorted(glob.glob(f"{shard_dir}/shard_*.npz"))
    assert paths, f"no shards at {shard_dir}"
    zs, ms, acts, sizes = [], [], [], None
    metas = {k: [] for k in META_KEYS}
    for p in tqdm(paths, desc=f"load {pathlib.Path(shard_dir).name}"):
        a = np.load(p)
        zs.append(a["prefix_out"]); ms.append(a["prefix_mask"]); acts.append(a["actions"])
        for k in META_KEYS:
            if k in a.files:
                metas[k].append(a[k])
        if sizes is None:
            sizes = {k: int(a[k]) for k in ["tokens_per_cam", "n_prompt", "n_reasoning", "n_subtask", "n_fast"]}
    z = np.concatenate(zs); m = np.concatenate(ms); act = np.concatenate(acts)
    meta = {k: np.concatenate(v) for k, v in metas.items() if v}
    if max_samples:
        z, m, act = z[:max_samples], m[:max_samples], act[:max_samples]
        meta = {k: v[:max_samples] for k, v in meta.items()}
    return z, m, act, sizes, meta

# Point at TRAIN_DIR for the training-set analysis (switch to VAL_DIR if desired).
z, m, act, sizes, meta = load_shards(str(TRAIN_DIR), MAX_SAMPLES)
print("z:", z.shape, z.dtype, "| mask density: %.3f" % m.mean(), "| actions:", act.shape)
print("metadata available:", list(meta.keys()) or "(none -- re-dump with --save-metadata)")

# Segment layout of the compacted dump (see scripts/dump_rl_token_embeddings.py).
seg_lens = [("vision", sizes["tokens_per_cam"]), ("prompt", sizes["n_prompt"]),
            ("reasoning", sizes["n_reasoning"]), ("subtask", sizes["n_subtask"]), ("fast", sizes["n_fast"])]
segments, s = [], 0
for name, ln in seg_lens:
    segments.append((name, s, s + ln)); s += ln
assert s == z.shape[1], (s, z.shape[1])
print("segments:", segments)

### What is a single data point?

One row = one **frame** (a single timestep from a driving episode). Episodes and dump-time batches are flattened away, so every row is an independent frame. A data point consists of:

| field | shape | meaning |
|---|---|---|
| `z` (`prefix_out`) | `(M, D)` | the VLM prefix embedding sequence `z_{1:M}` — **the AE's input and reconstruction target** |
| `m` (`prefix_mask`) | `(M,)` | which of the `M` tokens are real vs padding |
| `act` (`actions`) | `(H, A)` | the future action chunk the VLA predicts (carried along; **not** seen by the AE) |
| `meta[...]` | per-field | optional raw inputs that *produced* `z`: tokenized prompt / reasoning / subtask / fast (+ masks) and the base-camera frame (JPEG) |

The `M` tokens are laid out in fixed segments `[vision | prompt | reasoning | subtask | fast]` (lengths in `sizes`). The AE compresses this whole sequence into the single token `z_rl` and reconstructs `z_{1:M}`; **every metric in this notebook is per data point (per frame).**

In [ ]:
# Anatomy of one data point (row 0).
i = 0
print(f"data point {i}: one frame (timestep) from a driving episode\n")
print(f"  z   (prefix_out): {tuple(z[i].shape)}  {z.dtype}   -> M tokens x D dims  (AE input/target)")
print(f"  m   (prefix_mask): {tuple(m[i].shape)}  valid tokens = {int(m[i].sum())}/{m.shape[1]}")
print(f"  act (actions)    : {tuple(act[i].shape)}   -> future action chunk (not seen by AE)")
print("\n  token segments  [start:end]  valid_count:")
for name, a, b in segments:
    print(f"    {name:9s} [{a:3d}:{b:3d}]   {int(m[i, a:b].sum())}")
print("\n  metadata fields:", list(meta.keys()) or "(none -- dump without --save-metadata)")

## 3. Run the AE and collect per-sample metrics

For each sample we compute the masked-mean L2 reconstruction loss, the RL token `z_rl`, and the masked per-token L2.

In [ ]:
@torch.no_grad()
def run_metrics(z, m, bs=64):
    loss, zrl, ptl = [], [], []
    for i in tqdm(range(0, z.shape[0], bs), desc="infer"):
        zb = torch.from_numpy(z[i:i + bs]).to(device).float()
        mb = torch.from_numpy(m[i:i + bs]).to(device)
        out = model(zb, mb)
        mf = mb.float()
        per = out["per_token_l2"]                          # (b, M) unmasked
        ls = (per * mf).sum(1) / mf.sum(1).clamp(min=1)    # per-sample masked-mean L2
        loss.append(ls.cpu().numpy())
        zrl.append(out["z_rl"].float().cpu().numpy())
        ptl.append((per * mf).cpu().numpy())               # masked per-token L2
    return np.concatenate(loss), np.concatenate(zrl), np.concatenate(ptl)

loss, zrl, ptl = run_metrics(z, m)
print("per-sample loss: mean=%.2f median=%.2f min=%.2f max=%.2f"
      % (loss.mean(), np.median(loss), loss.min(), loss.max()))

# One fixed subsample, reused by every distributional cell below (§10/§16/§17/§18) so their
# numbers are mutually comparable and reproducible. Shrink SUBSAMPLE to speed up the whole notebook.
SUBSAMPLE = min(8000, len(loss))
SUB_IDX = np.sort(np.random.default_rng(0).choice(len(loss), SUBSAMPLE, replace=False))

## 4. Reconstruction-loss distribution

Histogram of per-sample loss, plus the indices of the lowest- and highest-loss samples for later inspection.

In [ ]:
k = 8
order = np.argsort(loss)
low_idx, high_idx = order[:k], order[-k:][::-1]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(loss, bins=60, color="steelblue")
ax[0].set_title("per-sample recon loss"); ax[0].set_xlabel("masked L2")
ax[1].hist(np.log10(loss + 1e-9), bins=60, color="indianred")
ax[1].set_title("log10 loss"); ax[1].set_xlabel("log10 L2")
plt.tight_layout(); plt.show()

print("lowest-loss idx :", low_idx, "\n  vals:", np.round(loss[low_idx], 2))
print("highest-loss idx:", high_idx, "\n  vals:", np.round(loss[high_idx], 2))

## 5. Where does the loss live? (per segment / per token)

Which parts of the prefix (vision / prompt / reasoning / subtask / fast) reconstruct well, and how the per-token profile differs between low- and high-loss samples.

In [ ]:
# Per-segment masked-mean L2 (averaged over all valid tokens in that segment).
seg_loss = {name: ptl[:, a:b].sum() / max(m[:, a:b].sum(), 1) for name, a, b in segments}
plt.figure(figsize=(7, 4))
plt.bar(list(seg_loss.keys()), list(seg_loss.values()), color="slateblue")
plt.ylabel("mean masked L2 / token"); plt.title("reconstruction loss by segment"); plt.show()

# Per-token mean across samples (valid tokens only): all vs low-loss 10% vs high-loss 10%.
qlo, qhi = np.quantile(loss, 0.1), np.quantile(loss, 0.9)
g_lo, g_hi = loss <= qlo, loss >= qhi
def tok_mean(grp):
    return ptl[grp].sum(0) / np.clip(m[grp].sum(0), 1, None)
tm_all, tm_lo, tm_hi = tok_mean(np.ones(len(loss), bool)), tok_mean(g_lo), tok_mean(g_hi)

plt.figure(figsize=(14, 4))
plt.plot(tm_all, label="all", lw=1, color="gray")
plt.plot(tm_lo, label="low-loss 10%", lw=1, color="green")
plt.plot(tm_hi, label="high-loss 10%", lw=1, color="red")
ymax = plt.ylim()[1]
for name, a, b in segments:
    plt.axvspan(a, b, alpha=0.05, color="blue")
    plt.text((a + b) / 2, ymax * 0.92, name, ha="center", fontsize=8)
plt.xlabel("token position"); plt.ylabel("mean L2"); plt.legend()
plt.title("per-token reconstruction loss"); plt.show()

### Spatial map of vision reconstruction loss

The vision segment is a square grid of patch tokens (e.g. 256 = 16×16) in raster order, so averaging the per-token loss across samples and reshaping back to the grid shows *where in the frame* the AE reconstructs well vs poorly — and whether high-loss samples differ spatially from low-loss ones.

In [ ]:
# Vision tokens -> square grid (raster order). ptl is masked per-token L2; vision tokens are
# always valid, so a plain mean over samples gives the spatial loss pattern.
va, vb = segments[0][1], segments[0][2]
G = int(round(sizes["tokens_per_cam"] ** 0.5))
assert G * G == sizes["tokens_per_cam"], f"vision tokens {sizes['tokens_per_cam']} not a square grid"

vis = ptl[:, va:vb]                                    # (N, G*G)
qlo, qhi = np.quantile(loss, 0.1), np.quantile(loss, 0.9)
grids = [("low-loss 10% samples", vis[loss <= qlo].mean(0).reshape(G, G)),
         ("all samples",          vis.mean(0).reshape(G, G)),
         ("high-loss 10% samples", vis[loss >= qhi].mean(0).reshape(G, G))]

fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
for a, (t, g) in zip(ax, grids):
    im = a.imshow(g, cmap="magma"); a.set_title(f"vision recon L2\n{t}")
    a.set_xticks([]); a.set_yticks([]); plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.show()

gm = grids[1][1]
print("vision-grid L2 (all samples): min=%.2f max=%.2f  argmax(row,col)=%s"
      % (gm.min(), gm.max(), np.unravel_index(gm.argmax(), gm.shape)))

## 6. RL-token latent space

Project `z_rl` to 2D (PCA, optional t-SNE) colored by reconstruction loss. Are high-loss samples outliers / clustered in latent space?

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(zrl)
emb2 = pca.transform(zrl)
c = np.log10(loss + 1e-9)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sc = ax[0].scatter(emb2[:, 0], emb2[:, 1], c=c, cmap="viridis", s=8)
ax[0].set_title("PCA of z_rl (color = log10 loss)")
ax[0].set_xlabel("PC1"); ax[0].set_ylabel("PC2"); plt.colorbar(sc, ax=ax[0])
print("PCA explained var:", np.round(pca.explained_variance_ratio_, 3))

# Optional t-SNE (color = log10 loss). t-SNE is ~O(N^2); on tens of thousands of 2048-d points it is
# very slow, so we subsample and pre-reduce to 50 PCA dims (standard practice -> large speedup).
try:
    from sklearn.manifold import TSNE
    TSNE_MAX = 8000
    tsne_idx = np.random.default_rng(0).choice(len(zrl), min(TSNE_MAX, len(zrl)), replace=False)
    z50 = PCA(n_components=min(50, zrl.shape[1])).fit_transform(zrl[tsne_idx])
    ts = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(z50)
    sc2 = ax[1].scatter(ts[:, 0], ts[:, 1], c=c[tsne_idx], cmap="viridis", s=8)
    ax[1].set_title(f"t-SNE of z_rl ({len(tsne_idx)} sampled)"); plt.colorbar(sc2, ax=ax[1])
except Exception as e:
    ax[1].set_title(f"t-SNE skipped: {e}")
plt.tight_layout(); plt.show()

## 7. Inspect individual low- vs high-loss samples

Reconstruct one low-loss and one high-loss sample; show the abs-error heatmap and per-token loss, plus segment lengths and action magnitude.

In [ ]:
@torch.no_grad()
def reconstruct(idx):
    zb = torch.from_numpy(z[idx:idx + 1]).to(device).float()
    mb = torch.from_numpy(m[idx:idx + 1]).to(device)
    return model(zb, mb)["z_hat"][0].float().cpu().numpy()

def seg_lengths(idx):
    return {name: int(m[idx, a:b].sum()) for name, a, b in segments}

def show_sample(idx, title):
    zt, zh, mk = z[idx].astype(np.float32), reconstruct(idx), m[idx]
    err = np.abs(zt - zh)                                   # (M tokens, D dims)
    fig, ax = plt.subplots(1, 2, figsize=(15, 4))
    # heatmap of the first 256 of D embed dims (plotting all 2048 is unreadable); rows=dim, cols=token.
    im = ax[0].imshow(err[:, :256].T, aspect="auto", cmap="magma"); plt.colorbar(im, ax=ax[0])
    ax[0].set_title(f"{title} | |z - z_hat| (first 256 dims)"); ax[0].set_xlabel("token"); ax[0].set_ylabel("dim")
    per = ((zt - zh) ** 2).sum(-1) * mk
    ax[1].plot(per, color="crimson"); ax[1].set_title(f"{title} | per-token L2 (loss={loss[idx]:.1f})")
    for name, a, b in segments:
        ax[1].axvspan(a, b, alpha=0.05, color="blue")
    ax[1].set_xlabel("token")
    plt.tight_layout(); plt.show()
    print(f"{title} idx={idx} valid-lengths={seg_lengths(idx)} action|mean|={np.abs(act[idx]).mean():.3f}")

show_sample(int(low_idx[0]), "LOW loss")
show_sample(int(high_idx[0]), "HIGH loss")

## 8. What correlates with high loss?

Scatter loss against simple sample features (reasoning / subtask / fast valid lengths, total valid tokens, action magnitude) with Pearson `r`.

In [ ]:
seg = {name: (a, b) for name, a, b in segments}
feat = {
    "reasoning_len":  m[:, seg["reasoning"][0]:seg["reasoning"][1]].sum(1),
    "subtask_len":    m[:, seg["subtask"][0]:seg["subtask"][1]].sum(1),
    "fast_len":       m[:, seg["fast"][0]:seg["fast"][1]].sum(1),
    "total_valid":    m.sum(1),
    "action_absmean": np.abs(act).reshape(len(loss), -1).mean(1),
}
fig, axs = plt.subplots(1, len(feat), figsize=(4 * len(feat), 4))
for ax, (name, v) in zip(axs, feat.items()):
    v = v.astype(np.float64)
    ax.scatter(v, loss, s=6, alpha=0.4)
    r = np.corrcoef(v, loss)[0, 1] if v.std() > 0 else float("nan")
    ax.set_title(f"{name}\nr={r:.2f}"); ax.set_xlabel(name); ax.set_ylabel("loss")
plt.tight_layout(); plt.show()

### Reading the results so far / what's next

- **Loss** is the per-sample *masked-mean* L2: the model's `per_token_l2` is unmasked, so we multiply by the prefix mask to drop padded slots before averaging.
- Raw L2 isn't interpretable on its own. The remaining sections normalize it against a **mean-predictor baseline** (§10) and split direction vs magnitude error via cosine — a loss well below that baseline means `z_rl` carries real information.
- The next sections inspect the **raw inputs** (text via §9, images via §13–§15) and the **`z_rl` token itself** (§11 geometry, §12 action information). These require shards dumped with `--save-metadata` / `--save-images`; on embedding-only dumps they no-op with a hint.
- **Different run / split:** point `RUN`/`CKPT` (and `TRAIN_DIR`→`VAL_DIR`) at another checkpoint and re-run top-to-bottom.

## 9. View the raw input behind each loss

Requires shards dumped with `--save-metadata` (token ids → text) and `--save-images` (base_0_rgb). Each row's image/text lives in the same shard row as its embedding, so the raw input ↔ embedding ↔ loss mapping is exact (shuffle-proof).

In [ ]:
# Detokenizer (PaliGemma sentencepiece). Reserved/CoT ids (>= base vocab) are shown as <id>.
import sentencepiece
from openpi.shared import download

_tok_path = download.maybe_download("gs://big_vision/paligemma_tokenizer.model", gs={"token": "anon"})
sp = sentencepiece.SentencePieceProcessor(model_proto=open(_tok_path, "rb").read())
BASE_VOCAB = sp.get_piece_size()

def detok(ids, mask=None):
    ids = np.asarray(ids).ravel().tolist()
    if mask is not None:
        mk = np.asarray(mask).ravel().astype(bool).tolist()
        ids = [t for t, k in zip(ids, mk) if k]
    out, buf = [], []
    for t in ids:
        if 0 <= int(t) < BASE_VOCAB:
            buf.append(int(t))
        else:
            if buf: out.append(sp.decode(buf)); buf = []
            out.append(f"<{int(t)}>")
    if buf: out.append(sp.decode(buf))
    return " ".join(out).strip()

In [ ]:
import io
from PIL import Image

# JPEG frames are stored as one flat byte buffer + per-row lengths; build a row -> byte offset map.
_JPEG_OFF = (np.concatenate([[0], np.cumsum(meta["base_image_jpeg_lengths"])])
             if "base_image_jpeg" in meta else None)

def get_image(idx):
    """Decode sample idx's base_0_rgb frame (JPEG blob or raw uint8); None if no image was dumped."""
    if "base_image" in meta:
        return meta["base_image"][idx]
    if _JPEG_OFF is not None:
        a, b = int(_JPEG_OFF[idx]), int(_JPEG_OFF[idx + 1])
        return np.array(Image.open(io.BytesIO(meta["base_image_jpeg"][a:b].tobytes())))
    return None

def _mask_for(key, idx):
    mk = key + "_mask"
    return meta[mk][idx] if mk in meta else None

def show_input(idx):
    print(f"idx={idx}  loss={loss[idx]:.2f}  action|mean|={np.abs(act[idx]).mean():.3f}")
    for key, label in [("tokenized_prompt", "PROMPT "), ("tokenized_reasoning", "REASON "),
                       ("tokenized_subtask", "SUBTASK"), ("tokenized_fast", "FAST   ")]:
        if key in meta:
            print(f"  {label}:", detok(meta[key][idx], _mask_for(key, idx)))
    img = get_image(idx)
    if img is not None:
        plt.figure(figsize=(4, 4)); plt.imshow(img); plt.axis("off")
        plt.title(f"idx {idx}  loss {loss[idx]:.1f}"); plt.show()

if not meta:
    print("No metadata in these shards. Re-dump with --save-metadata (and --save-images) to enable this.")
else:
    print("=== LOWEST-loss inputs ===")
    for i in low_idx[:3]:
        show_input(int(i))
    print("\n=== HIGHEST-loss inputs ===")
    for i in high_idx[:3]:
        show_input(int(i))

## 10. Is the loss actually good? (normalized loss + cosine)

Raw L2 is hard to read. Here we compare it to a **mean-predictor baseline** (reconstruct every token as the dataset-mean embedding): `normalized = recon_L2 / baseline_L2`, where `1.0` means "no better than the mean" and `0` means perfect. We also report **cosine(z, z_hat)** per segment to separate direction errors from magnitude errors.

In [ ]:
import torch.nn.functional as F

# Distributional stats on the shared subsample; cosine accumulated on-device (one host sync at the end).
sub, SUB = SUB_IDX, SUBSAMPLE
bs, D = 256, z.shape[-1]

# pass 1 (CPU): mean embedding over valid tokens of the subsample.
s, cnt = np.zeros(D, np.float64), 0.0
for i in range(0, SUB, bs):
    idx = sub[i:i + bs]
    zb = z[idx].astype(np.float32); mf = m[idx].astype(np.float32)
    s += (zb * mf[..., None]).sum((0, 1)); cnt += mf.sum()
mean_z = (s / cnt).astype(np.float32)

# pass 2 (GPU): per-sample baseline (predict mean) + cosine(z, z_hat) per segment.
mz = torch.from_numpy(mean_z).to(device)
cos_sum = torch.zeros(len(segments), device=device); cos_cnt = torch.zeros(len(segments), device=device)
baseline = np.empty(SUB, np.float32)
with torch.no_grad():
    for i in range(0, SUB, bs):
        idx = sub[i:i + bs]
        zb = torch.from_numpy(z[idx].astype(np.float32)).to(device)
        mb = torch.from_numpy(m[idx]).to(device); mf = mb.float()
        bl = (((zb - mz) ** 2).sum(-1) * mf).sum(1) / mf.sum(1).clamp(min=1)
        baseline[i:i + bs] = bl.cpu().numpy()
        cos = F.cosine_similarity(zb, model(zb, mb)["z_hat"], dim=-1)
        for si, (_, a, b) in enumerate(segments):
            cos_sum[si] += (cos[:, a:b] * mf[:, a:b]).sum(); cos_cnt[si] += mf[:, a:b].sum()
cos_vals = (cos_sum / cos_cnt.clamp(min=1)).cpu().numpy()
cos_seg = {n: float(cos_vals[si]) for si, (n, _, _) in enumerate(segments)}
norm = loss[sub] / np.clip(baseline, 1e-9, None)

print("aggregate R^2 (1 - recon/baseline): %.3f   [n=%d subsample]" % (1 - loss[sub].mean() / baseline.mean(), SUB))
print("normalized loss: mean=%.3f median=%.3f (1.0 = mean-predictor)" % (norm.mean(), np.median(norm)))
print("cosine(z, z_hat) by segment:", {k: round(v, 3) for k, v in cos_seg.items()})

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(norm, bins=60, color="teal"); ax[0].axvline(1, color="k", ls="--", lw=1)
ax[0].set_title("normalized loss (recon L2 / baseline L2)"); ax[0].set_xlabel("lower = better; 1.0 = mean-predictor")
ax[1].bar(list(cos_seg.keys()), list(cos_seg.values()), color="darkorange")
ax[1].set_ylim(0, 1); ax[1].set_ylabel("cosine(z, z_hat)"); ax[1].set_title("reconstruction cosine by segment")
plt.tight_layout(); plt.show()

## 11. `z_rl` health — is the bottleneck using its capacity?

`z_rl` is the single token you'll hand to RL, so its geometry matters. We look at the **norm distribution**, the **per-dim variance**, and how concentrated the variance is across PCA directions (**participation ratio**, components for 90/99%).

**Caveat — these are variance-weighted descriptors, not a measure of useful capacity.** PCA down-weights low-variance directions that the decoder may still rely on, and the near-constant `||z_rl||` is largely an artifact of the encoder's final `LayerNorm` (so a high cosine between any two `z_rl` reflects a shared mean component, not collapse). Read a low participation ratio as *anisotropy*, not "the bottleneck is wasted." Whether the extra dimensions actually reduce reconstruction error is tested directly in **§18** (decode loss vs. number of `z_rl` PCs retained).

In [ ]:
znorm = np.linalg.norm(zrl, axis=1)
Xc = zrl - zrl.mean(0)
ev = np.clip(np.linalg.eigvalsh(Xc.T @ Xc / len(Xc))[::-1], 0, None)   # covariance spectrum
cum = np.cumsum(ev / ev.sum())
pr = ev.sum() ** 2 / (ev ** 2).sum()                                   # participation ratio
eff90, eff99 = int(np.searchsorted(cum, 0.9) + 1), int(np.searchsorted(cum, 0.99) + 1)
dim_var = zrl.var(0); dead = int((dim_var < 0.01 * dim_var.max()).sum())

print("z_rl dim=%d | participation ratio=%.1f | comps for 90%%=%d, 99%%=%d | near-dead dims=%d"
      % (zrl.shape[1], pr, eff90, eff99, dead))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(znorm, bins=60, color="steelblue")
ax[0].set_title("||z_rl|| distribution"); ax[0].set_xlabel("L2 norm")
ax[1].plot(cum, color="purple"); ax[1].axhline(0.9, color="k", ls="--", lw=1); ax[1].axvline(eff90, color="red", ls=":")
ax[1].set_xlim(0, min(512, len(cum))); ax[1].set_ylim(0, 1.01)
ax[1].set_title("cumulative explained variance"); ax[1].set_xlabel("# PCA components")
ax[2].plot(np.sort(dim_var)[::-1], color="green"); ax[2].set_yscale("log")
ax[2].set_title("per-dim variance (sorted)"); ax[2].set_xlabel("dim rank"); ax[2].set_ylabel("variance")
plt.tight_layout(); plt.show()

## 12. Linear probe: how much action info survives in `z_rl`?

The whole point of RLT is that the single `z_rl` token carries the decision-relevant content of the sequence. We fit a ridge regression from `z_rl → action` (held-out R²) and compare it to probing the **mean-pooled prefix** (an upper bound on what's available in the full sequence). A small gap means the bottleneck preserves the action-relevant information.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

Y = act.reshape(len(act), -1).astype(np.float64)
H_, A_ = (act.shape[1], act.shape[2]) if act.ndim == 3 else (1, act.shape[1])

# mean-pooled prefix (info-available reference), batched to avoid a float32 copy of z.
pooled = np.zeros((len(z), z.shape[-1]), np.float32)
for i in range(0, len(z), 256):
    zb = z[i:i + 256].astype(np.float32); mf = m[i:i + 256].astype(np.float32)
    pooled[i:i + 256] = (zb * mf[..., None]).sum(1) / mf.sum(1, keepdims=True).clip(min=1)

def probe(X):
    # Standardize first so the shared ridge penalty is fair: z_rl is LayerNorm'd (~unit scale) while the
    # pooled prefix has large, uneven per-dim norms -- a fixed alpha on raw features would favor pooled.
    Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.2, random_state=0)
    p = make_pipeline(StandardScaler(), Ridge(alpha=10.0)).fit(Xtr, Ytr).predict(Xte)
    r2 = 1 - ((Yte - p) ** 2).sum(0) / np.clip(((Yte - Yte.mean(0)) ** 2).sum(0), 1e-9, None)
    return r2.reshape(H_, A_).mean(0)                     # avg over horizon -> per action component

r2_zrl, r2_pool = probe(zrl), probe(pooled)
info = r2_pool > 0.05                                     # components the full prefix can actually predict
print("action probe R^2 | z_rl:", np.round(r2_zrl, 3))
print("action probe R^2 | pool:", np.round(r2_pool, 3))
print("informative components (pool R^2>0.05): %d/%d | median R^2  z_rl=%.3f  pool=%.3f"
      % (int(info.sum()), len(info),
         float(np.median(r2_zrl[info])) if info.any() else float("nan"),
         float(np.median(r2_pool[info])) if info.any() else float("nan")))

x, w = np.arange(A_), 0.4
plt.figure(figsize=(max(6, A_ * 1.1), 4))
plt.bar(x - w / 2, r2_zrl, w, label="z_rl (bottleneck)", color="crimson")
plt.bar(x + w / 2, r2_pool, w, label="mean-pooled prefix", color="gray")
plt.xticks(x, [f"action[{i}]" for i in x]); plt.ylabel("held-out R^2"); plt.ylim(0, 1)
plt.title("linear-probe action recovery: z_rl vs full sequence"); plt.legend(); plt.show()

## 13. Loss by scenario (decoded prompt text)

Group samples by keywords in the decoded prompt (turn / straight / stop / …) and compare mean reconstruction loss per bucket against the overall mean. Surfaces *which driving situations* the AE struggles to compress. (A sample can match multiple keywords.)

In [ ]:
if "tokenized_prompt" not in meta:
    print("no tokenized_prompt metadata; re-dump with --save-metadata")
else:
    pm = meta.get("tokenized_prompt_mask")
    texts = [detok(meta["tokenized_prompt"][i], None if pm is None else pm[i]).lower()
             for i in tqdm(range(len(loss)), desc="detok prompts")]
    keywords = ["left", "right", "straight", "stop", "slow", "follow", "lane",
                "merge", "traffic light", "pedestrian", "park"]
    rows = []
    for kw in keywords:
        idxs = [i for i, t in enumerate(texts) if kw in t]
        if idxs:
            rows.append((kw, len(idxs), float(np.mean(loss[idxs]))))
    rows.sort(key=lambda r: -r[2])
    print("scenario keyword | count | mean loss  (overall=%.2f)" % loss.mean())
    for kw, n, ml in rows:
        print(f"  {kw:14s} {n:6d}  {ml:.2f}")
    plt.figure(figsize=(10, 4))
    plt.bar([r[0] for r in rows], [r[2] for r in rows], color="slateblue")
    plt.axhline(loss.mean(), color="k", ls="--", lw=1, label="overall mean")
    plt.ylabel("mean recon loss"); plt.title("loss by scenario keyword")
    plt.xticks(rotation=45, ha="right"); plt.legend(); plt.tight_layout(); plt.show()

## 14. Nearest neighbors in `z_rl` space

For a query sample, show the frames of its closest neighbors by cosine similarity in `z_rl`. Reveals what the compressed token organizes by — visual scene, action, or scenario.

In [ ]:
_zn = zrl / (np.linalg.norm(zrl, axis=1, keepdims=True) + 1e-9)

def show_neighbors(q, k=5):
    sim = _zn @ _zn[q]
    nn = np.argsort(-sim)[:k + 1]                         # includes the query itself
    fig, ax = plt.subplots(1, len(nn), figsize=(2.6 * len(nn), 2.9))
    for a, j in zip(ax, nn):
        img = get_image(int(j))
        if img is not None:
            a.imshow(img)
        a.axis("off")
        a.set_title("QUERY" if j == q else f"cos {sim[j]:.2f}\nloss {loss[j]:.1f}", fontsize=8)
    plt.suptitle(f"z_rl nearest neighbors of idx {q}"); plt.tight_layout(); plt.show()

if "base_image_jpeg" in meta or "base_image" in meta:
    show_neighbors(int(high_idx[0]))
    show_neighbors(int(low_idx[0]))
else:
    print("no images in shards; re-dump with --save-images")

## 15. Vision loss overlaid on a real frame

Upsample the vision loss grid (G×G → image size) and alpha-blend it over the actual frame, so the spatial loss pattern is interpretable in context. Shows both the per-sample grid and the dataset-mean grid for a high-loss sample.

In [ ]:
def overlay_vision_loss(idx, grid, title):
    img = get_image(int(idx))
    if img is None:
        print("no image for idx", idx); return
    g = (grid - grid.min()) / (np.ptp(grid) + 1e-9)
    H, W = img.shape[0], img.shape[1]
    up = np.kron(g, np.ones((max(H // G, 1), max(W // G, 1))))     # blocky G x G -> pixels
    plt.figure(figsize=(4.6, 4.6)); plt.imshow(img)
    plt.imshow(up, cmap="magma", alpha=0.45, extent=[0, W, H, 0])
    plt.axis("off"); plt.title(title); plt.colorbar(fraction=0.046); plt.show()

if "base_image_jpeg" in meta or "base_image" in meta:
    hi = int(high_idx[0])
    overlay_vision_loss(hi, vis[hi].reshape(G, G), f"idx {hi}: per-sample vision loss (loss={loss[hi]:.1f})")
    overlay_vision_loss(hi, gm, f"idx {hi} frame: dataset-mean vision loss")
else:
    print("no images in shards; re-dump with --save-images")

## 16. Per-dimension reconstruction loss

The AE reconstructs the VLM's latent embeddings (`D` dims); this decomposes the L2 across those feature dimensions. It complements §5 (loss per token/segment) and §7 (the dim-wise error streaks in the heatmap): **are a few feature dims dominating the loss, and is that just because they are high-variance?**

To de-confound magnitude we also report **per-dim normalized MSE = MSE / dim-variance = 1 − R²** (1.0 = no better than predicting that dim's global mean; lower is better). The `MSE vs variance` scatter makes the confound explicit: points on the `y=x` line are reconstructed no better than the mean, points below it carry real signal. Estimated on a random subsample for speed.

In [ ]:
# Accumulate, per embedding dim, the masked sum of squared error and the first/second moments of z
# (for the per-dim mean-predictor baseline). One model pass on a subsample; reductions stay on-device.
D = z.shape[-1]
sub, SUB = SUB_IDX, SUBSAMPLE
bs = 256

sse = torch.zeros(D, device=device)        # sum sq error per dim
sz = torch.zeros(D, device=device)         # sum of z per dim (valid tokens)
sz2 = torch.zeros(D, device=device)        # sum of z^2 per dim
cnt = torch.zeros(1, device=device)        # total valid tokens
with torch.no_grad():
    for i in range(0, SUB, bs):
        idx = sub[i:i + bs]
        zb = torch.from_numpy(z[idx].astype(np.float32)).to(device)
        mb = torch.from_numpy(m[idx]).to(device)
        mf = mb.float().unsqueeze(-1)                       # (b, M, 1)
        zh = model(zb, mb)["z_hat"]
        sse += (((zh - zb) ** 2) * mf).sum((0, 1))
        sz += (zb * mf).sum((0, 1))
        sz2 += ((zb ** 2) * mf).sum((0, 1))
        cnt += mf.sum()

mse_d = (sse / cnt).cpu().numpy()
var_d = (sz2 / cnt - (sz / cnt) ** 2).clamp(min=0).cpu().numpy()
nmse_d = mse_d / np.clip(var_d, 1e-12, None)               # per-dim 1 - R^2 (1.0 = mean-predictor)

order = np.argsort(mse_d)[::-1]
topk = 20
frac = mse_d[order[:topk]].sum() / mse_d.sum()
print("per-dim MSE: mean=%.3f max=%.3f at dim %d | top-%d dims hold %.1f%% of total loss"
      % (mse_d.mean(), mse_d.max(), int(order[0]), topk, 100 * frac))
print("per-dim normalized MSE (1 - R^2): mean=%.3f median=%.3f | dims worse than mean-predictor (>1): %d/%d"
      % (nmse_d.mean(), np.median(nmse_d), int((nmse_d > 1).sum()), D))
print("worst dims by raw MSE:", order[:10].tolist())

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(np.sort(mse_d)[::-1], color="crimson"); ax[0].set_yscale("log")
ax[0].set_title("per-dim recon MSE (sorted)"); ax[0].set_xlabel("dim rank"); ax[0].set_ylabel("MSE")
ax[1].hist(nmse_d, bins=60, color="teal"); ax[1].axvline(1, color="k", ls="--", lw=1)
ax[1].set_title("per-dim normalized MSE (1 - R^2)"); ax[1].set_xlabel("lower = better; 1.0 = mean-predictor")
pos = (var_d > 0) & (mse_d > 0)
ax[2].scatter(var_d[pos], mse_d[pos], s=5, alpha=0.3, color="slateblue")
lims = [min(var_d[pos].min(), mse_d[pos].min()), max(var_d.max(), mse_d.max())]
ax[2].plot(lims, lims, "k--", lw=1)                        # y=x: MSE == dim variance (mean-predictor)
ax[2].set_xscale("log"); ax[2].set_yscale("log")
ax[2].set_title("per-dim MSE vs variance"); ax[2].set_xlabel("dim variance"); ax[2].set_ylabel("dim MSE")
plt.tight_layout(); plt.show()

## 17. Dataset composition — what is the bulk (not the tails)?

Earlier sections inspected the **loss tails** (lowest/highest), which say nothing about what most frames look like. Here we look at **marginal distributions over a random subsample**: image **brightness** (a night/day proxy), **speed** parsed from the prompt, and **action magnitude**. This tests directly whether the dataset is dominated by a single "simple" mode (e.g. stopped/dark) or is broadly varied — and whether brightness/speed actually track loss dataset-wide, rather than only at the extremes. Requires `--save-metadata`/`--save-images`.

In [ ]:
import re

if not ("base_image_jpeg" in meta or "base_image" in meta) and "tokenized_prompt" not in meta:
    print("no metadata; re-dump with --save-metadata/--save-images to enable composition analysis")
else:
    SUBC = SUBSAMPLE
    subc = SUB_IDX
    bright = np.full(SUBC, np.nan); speed = np.full(SUBC, np.nan)
    for j, i in enumerate(tqdm(subc, desc="composition")):
        img = get_image(int(i))
        if img is not None:
            bright[j] = np.asarray(img).mean()
        if "tokenized_prompt" in meta:
            mo = re.search(r"speed is ([0-9.]+)", detok(meta["tokenized_prompt"][i], _mask_for("tokenized_prompt", i)))
            if mo:
                speed[j] = float(mo.group(1))
    amag = np.abs(act).reshape(len(act), -1).mean(1)[subc]
    lo = loss[subc]

    feats = [("brightness (mean pixel)", bright), ("speed (m/s)", speed), ("action |mean|", amag)]
    fig, ax = plt.subplots(2, len(feats), figsize=(15, 7))
    for c, (name, v) in enumerate(feats):
        ok = np.isfinite(v)
        ax[0, c].hist(v[ok], bins=50, color="steelblue"); ax[0, c].set_title(name); ax[0, c].set_xlabel(name)
        ax[1, c].scatter(v[ok], lo[ok], s=6, alpha=0.3, color="slateblue")
        r = np.corrcoef(v[ok], lo[ok])[0, 1] if ok.sum() > 2 and v[ok].std() > 0 else float("nan")
        ax[1, c].set_title(f"loss vs {name}  (r={r:.2f})"); ax[1, c].set_xlabel(name); ax[1, c].set_ylabel("loss")
    plt.tight_layout(); plt.show()

    print("marginal spread (a tight distribution => a dominant mode; a wide one => varied data):")
    for name, v in feats:
        ok = np.isfinite(v)
        if ok.sum():
            q = np.quantile(v[ok], [0.1, 0.5, 0.9])
            print("  %-22s n=%4d  p10=%.3f  median=%.3f  p90=%.3f" % (name, int(ok.sum()), q[0], q[1], q[2]))

## 18. Functional rank of `z_rl` — how many directions does the decoder actually need?

Participation ratio (§11) weights directions by *variance*, which can badly undercount low-variance directions the decoder still relies on. Here we measure **utility** instead: project `z_rl` onto its top-`k` PCA directions, feed the truncated code to the (teacher-forced) decoder, and watch reconstruction loss as a function of `k`. The curve from `k=0` (mean only) to `k=full` shows how many directions carry reconstruction-relevant information.

This is the principled test for "is the bottleneck width used?" — but note it is **not** the same as the capacity sweep over `d_model`, because in this architecture `d_model` sets the bottleneck width *and* the whole transformer's width, so a loss drop when widening `d_model` is confounded with simply having a bigger model. This test isolates the `z_rl` directions at fixed model size.

In [ ]:
SUBF = SUBSAMPLE
subf = SUB_IDX
zrl_s = zrl[subf].astype(np.float32)
mu = zrl_s.mean(0)
_, _, Vt = np.linalg.svd(zrl_s - mu, full_matrices=False)   # rows of Vt = principal directions

@torch.no_grad()
def decode_loss(zrl_trunc):
    tot, cnt = 0.0, 0.0
    for i in range(0, SUBF, 256):
        idx = subf[i:i + 256]
        zb = torch.from_numpy(z[idx].astype(np.float32)).to(device)
        mb = torch.from_numpy(m[idx]).to(device)
        zr = torch.from_numpy(zrl_trunc[i:i + 256]).to(device)
        per = ((model.decode(zr, zb, mb) - zb) ** 2).sum(-1) * mb.float()
        tot += per.sum().item(); cnt += mb.float().sum().item()
    return tot / cnt

D_ = zrl.shape[1]
ks = sorted({k for k in [0, 1, 2, 3, 4, 8, 16, 32, 64, 128, 256, 512, D_] if k <= D_})
losses_k = []
for k in ks:
    zk = np.repeat(mu[None], SUBF, 0) if k == 0 else mu + (zrl_s - mu) @ Vt[:k].T @ Vt[:k]
    losses_k.append(decode_loss(zk.astype(np.float32)))
full = losses_k[-1]
print("decode loss vs #PCs kept (full z_rl loss=%.2f, mean-only loss=%.2f):" % (full, losses_k[0]))
for k, l in zip(ks, losses_k):
    print("  k=%4d  loss=%7.2f  gap-closed=%.1f%%"
          % (k, l, 100 * (losses_k[0] - l) / max(losses_k[0] - full, 1e-9)))

plt.figure(figsize=(7, 4))
plt.plot(ks, losses_k, "o-", color="crimson")
plt.axhline(full, color="k", ls="--", lw=1, label="full z_rl")
plt.xscale("symlog"); plt.xlabel("# z_rl PCs kept"); plt.ylabel("decode recon loss")
plt.title("functional rank: decode loss vs z_rl PCs retained"); plt.legend(); plt.tight_layout(); plt.show()